# AIR proposal v2: explicit correspondence chain + Laplace mixture vs the pooled flow proposal

Issue #21. Spec: `docs/a2b_air_proposal_v2_2026-11-03.md` (Tim, 2026-11-03), incl. its *Refinement* section
(Gauss–Newton builds q, no MCMC; forward–backward; C_a widened by σ_A). Branch `exp/21-air-chain`.

## Description

<!-- Written before the first run (2026-11-03). -->

**Question.** Pilot 3's proposal q(θ | A, B) (pooled encoder + spline flow, notebook 11) underfits: the gap
KL(post ‖ q) grows with n_A (pilot-2 diagnostics: 0.28 / 0.94 / 1.08 nats at n_A = 1 / 5 / 10, J = 2). The
proposal v2 claim: a proposal built from **explicit per-point correspondence evidence** has no pooled bottleneck
and no branch averaging, so at equal K it should give higher ESS and a smaller NLL gap to the exact reference,
increasingly so for larger n_A and wigglier g. If true, the d ≥ 2 work becomes learned message passing over location
beliefs; if not, the bottleneck is elsewhere (decoder, or the transform prior itself).

**Setup.** Variant (c) of the spec, no learned parts (`experimental/air/chain.py`):
1. *B model*: a GP on B (RBF, ML over a lengthscale × noise grid) → predictive mean/var on a G = 256 grid of u ∈ [0, 1].
2. *Evidence volumes* C_a(u) = mean_s N(y_a; m_B(u), v_B(u) + s²) over 4 values s of σ_A's prior range
   (σ_A is latent; the fits use s = 0.02).
3. *Coherence*: 1-D chain over A sorted by x; transitions p(u_{a+1} | u_a) estimated from S = 4096 prior draws of θ
   (KDE, bandwidth 0.02); forward filtering, backward sampling of M = 128 correspondence sets.
4. *Fit*: per set, Levenberg–Marquardt for θ in whitened prior coordinates, first to the correspondences (τ = 0.03),
   then to the evidence objective; Laplace covariance (I + JᵀJ)⁻¹.
5. *Proposal*: q = (1 − ε)·(1/M) Σ_m N(θ̂_m, Σ_m) + ε·prior, ε = 0.1 (the same defensive ε as pilot 3's flow).

Compared at equal K ∈ {8, 32, 128} importance samples, with pilot 3's decoder (σ-conditioned, Σ = 0) as the
likelihood, on pilot 3's evaluation tasks (`eval_tasks(J, 256, 777, ecdf)`, J ∈ {2, 16}, n_A ∈ {1, 2, 3, 5, 10}):
- `flow` = (a): pilot 3's trained proposal of the same seed and J (with its ε-prior mixture);
- `flow_gn`: the flow's M = 128 draws as seeds → Gauss–Newton + Laplace mixture (refinement §1: the flow proposes,
  GN sharpens);
- `chain_gp` = (c);
- `chain_oracle` = (c) with the true g as B model (an upper bound on what better emissions, variant (d), could give);
- `grid` (J = 2 only): the exact dense-B Bayes reference on a 101² θ grid (pilot 3's G-ref).
For J = 16 there is no exact reference: methods are compared with each other and with pilot 3's `isref`
(K = 4096, same tasks, from snapshot `results/air_rb/<pilot-3 eval>`) by task id.
Seeds 0, 1, 2 = pilot 3's checkpoint seeds (decoder and proposals of that seed); the tasks are the same for all seeds.

**What to look for.**
- NLL(method, K) − NLL(grid) at J = 2 and NLL(flow) − NLL(chain_gp) at J = 2, 16, per n_A and K; paired by task.
- ESS/K per proposal; `logq_true` (density of q at the true θ); `rank_true` (share of q draws with higher q density
  than θ_true; uniform if q is calibrated, piled at 1 if q misses θ_true); KL(post ‖ q) at J = 2 (64 tasks).
- All of the above by the number of branches of g (monotone pieces), and the build cost per task.

**Pre-registered criteria** (paired mean over 256 tasks; a "cell" is (J, n_A); a criterion holds if it holds in each
of the 3 seeds):
- **V2.0 sanity**: at J = 2, n_A ≤ 3, chain_oracle NLL at K = 128 within 0.05 nats of pilot 3's `isref` (prior IS,
  K = 4096, the same decoder; joined by seed and task); chain_oracle KL(post ‖ q) ≤ chain_gp KL + 0.05 (better
  emissions do not hurt).
  *Amended 2026-11-03, before the first pilot run:* the original reference was `grid`, which uses the exact
  likelihood, so every proposal (also pilot 3's flow) sits ~0.15 nats above it because of the shared decoder
  (see Limits), and V2.0 could not pass whatever the proposal. `isref` has the same likelihood as the methods, so
  method − isref is the proposal's error alone. Noticed while preparing the analysis on the scratch probe tables;
  no pilot data had been produced. method − grid is still reported.
- **V2.1 primary**: at K = 32, NLL(flow) − NLL(chain_gp) ≥ 0.05 nats in at least 2 of the 3 cells n_A ∈ {3, 5, 10},
  for both J = 2 and J = 16.
- **V2.2**: ESS/K(chain_gp) > ESS/K(flow) at K = 32 in the same cells.
- **V2.1b** (reported, not gating): the same comparison for `flow_gn` vs `flow` (does GN + Laplace fix the flow's width?).
- **V2.3 n_A = 1**: |NLL(flow) − NLL(chain_gp)| ≤ 0.05 (the spec expects (a) to stay competitive; reported, not gating).
- **V2.4 branches**: the flow − chain_gp gap at n_A ≥ 3 increases with the branch count of g (reported, not gating).

**GO / NO-GO.** GO if V2.0, V2.1 and V2.2 hold: build the d ≥ 2 version (learned message passing over location
beliefs) and the J-sampled prior. NO-GO if NLL(flow) − NLL(chain_gp) < 0.02 in every cell with n_A ≥ 2 at
K = 32 (explicit correspondences buy nothing over the flow; the bottleneck is elsewhere). Otherwise INCONCLUSIVE.
If V2.0 fails, the result is INCONCLUSIVE regardless (the chain construction itself is broken).

**Limits.**
- 1-D, h = id, and **J known** (J = 2 or 16, fixed per run). The spec's setup samples J per task with a Laplace-evidence
  mixture over J; that needs a J-sampled training prior for decoder and flow. Staged next (isolation, not a scope cut).
- Only (a) and (c) (plus the oracle-emission bound). (b) transport-token loop + flow and (d) learned emissions are
  staged after this (they need training); `chain_oracle` bounds what (d) can gain.
- The likelihood is the trained decoder, not the exact one, so all methods share its error; at J = 2 `grid` uses the
  exact likelihood, so method − grid contains the decoder gap (~0.1–0.2 nats in pilot 2) in addition to the proposal gap.
  The comparison flow vs chain at equal K is unaffected.
- The chain's transitions use a KDE over prior samples: fine in 1-D, no claim about d ≥ 2.
- Pilot only (split 0 tasks, seed 777); nothing here is tuned on split 1.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/air_chain/<sweep_id>"  # set after `just snapshot runs/air_chain/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.